# DistLLM - Kaggle worker joining your laptop's cluster (v3)
This notebook runs a real DistLLM **worker** on the free Kaggle T4x2 and
joins your laptop as a second node in the cluster. The laptop's
coordinator assigns the layer split, so the Llama model literally
lives across both boxes.

## What this proves
1. Real **pipeline parallelism** across the public internet: laptop (8 GB VRAM)
   and Kaggle GPU (16 GB VRAM) = **24 GB combined** holding one model.
2. The laptop's coordinator auto-assigns layers (e.g. 0-15 on laptop,
   16-31 on Kaggle for Llama-3.1-8B's 32 layers).
3. The laptop's REST API then serves OpenAI-compatible requests that
   cross both GPUs.

## Before you run
1. On your laptop, start the demo (uses `bore` to expose gRPC):
```powershell
$env:DISTLLM_DEMO_MODEL    = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
$env:DISTLLM_DEMO_API_KEY  = 'demo-laptop-2026'
$env:HUGGING_FACE_HUB_TOKEN = 'hf_your_token_here'
python distllm_laptop_coordinator.py
```
2. In Kaggle: **Add-ons -> Secrets** -> add `KAGGLE_USERNAME` and `KAGGLE_KEY`
   (Kaggle uses its own gated-model credentials, NOT `HF_TOKEN`).
   For non-gated models you can skip the secrets.
3. Copy the laptop's bore.pub host/port/cluster key into cell #2 below.
4. Run the rest of the cells in order.


In [ ]:
#@title 1. Install DistLLM with CUDA support
!pip install --quiet --upgrade 'distllm[cuda]' 2>/dev/null || !pip install --quiet --upgrade 'distllm[gpu]' transformers accelerate

import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
    print('vram  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


## 2. Laptop coordinator address


In [ ]:
#@title 2. Paste the laptop coordinator address and cluster key
import os
from kaggle_secrets import UserSecretsClient  # Kaggle-only

DISTLLM_PUBLIC_HOST = 'bore.pub'  #@param {type:'string'}
DISTLLM_PUBLIC_PORT = '17128'      #@param {type:'string'}
CLUSTER_KEY         = 'demo-cluster-key'  #@param {type:'string'}
NODE_ID             = 'kaggle-t4'  #@param {type:'string'}
MODEL               = 'meta-llama/Meta-Llama-3.1-8B-Instruct'  #@param {type:'string'}

# TOTAL_LAYERS for Llama-3.1-8B-Instruct is 32. The laptop's coordinator
# will auto-assign half to each node (L=0-15 on laptop, L=16-31 on Kaggle)
# when start_layer / end_layer are 0 (the worker leaves assignment to the
# coordinator).
START_LAYER  = 0   #@param {type:'integer'}
END_LAYER    = 0   #@param {type:'integer'}
TOTAL_LAYERS = 32  #@param {type:'integer'}
LISTEN_PORT  = 50051  #@param {type:'integer'}

# Kaggle uses its OWN gated-model credentials (KAGGLE_USERNAME / KAGGLE_KEY),
# exposed via kaggle_secrets. We forward them as the standard HF env vars
# so distllm.dist.worker can pull gated weights the same way it does on
# Colab. If you don't need gated weights, this block is a no-op.
try:
    _secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = _secrets.get_secret('KAGGLE_KEY')
    # distllm.dist.worker reads HF_TOKEN for gated HuggingFace models;
    # mirror the Kaggle creds into HF_TOKEN for compatibility.
    os.environ['HF_TOKEN']        = os.environ['KAGGLE_KEY']
    print('loaded KAGGLE_USERNAME / KAGGLE_KEY from Kaggle secrets')
except Exception as _e:
    print('no Kaggle secrets available:', _e, '- skipping gated-model auth')

os.environ['DISTLLM_CLUSTER_KEY'] = CLUSTER_KEY
print('joining cluster at', DISTLLM_PUBLIC_HOST + ':' + DISTLLM_PUBLIC_PORT)
print('node_id', NODE_ID, '| model', MODEL, '| layers', f'{START_LAYER}-{END_LAYER}/{TOTAL_LAYERS}')


In [ ]:
#@title 3. Start the worker (talks to your laptop via bore.pub)
import os, sys, subprocess, signal

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
    print('WARNING: no CUDA, falling back to CPU (very slow)')

worker = subprocess.Popen(
    [
        sys.executable, '-m', 'distllm.dist.worker',
        '--node-id',            NODE_ID,
        '--model',              MODEL,
        '--start-layer',        str(START_LAYER),
        '--end-layer',          str(END_LAYER),
        '--total-layers',       str(TOTAL_LAYERS),
        '--port',               str(LISTEN_PORT),
        '--coordinator-host',   DISTLLM_PUBLIC_HOST,
        '--coordinator-port',   DISTLLM_PUBLIC_PORT,
        '--device',             device,
    ],
    env={**os.environ, 'DISTLLM_CLUSTER_KEY': CLUSTER_KEY},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print(f'worker PID {worker.pid}, talking to {DISTLLM_PUBLIC_HOST}:{DISTLLM_PUBLIC_PORT}')
print('streaming worker logs (Stop to interrupt):')
try:
    for line in iter(worker.stdout.readline, ''):
        print(line, end='')
        if 'listening on' in line.lower() or 'ready' in line.lower():
            print('\n--- worker is ready; the cluster is now live ---')
except KeyboardInterrupt:
    worker.send_signal(signal.SIGTERM)


## 4. Test the cluster from Kaggle (optional)
The worker is now part of the laptop's cluster. Hit the laptop's REST API
directly (much faster than going through the gRPC worker) to see the
laptop's half of the model respond.


In [ ]:
#@title 4. Quick chat test against the laptop's REST API
import os
LAPTOP_API = 'http://localhost:8000'  # laptop's REST API; not tunneled
# If you want to hit the laptop from Kaggle you need a tunnel; easiest is
# to run the chat from the laptop terminal directly.
print('Run this from the laptop terminal instead, where the REST API is at:')
print('  http://localhost:8000/v1/chat/completions')
print()
print('Or expose it with bore on the laptop too:')
print('  bore local 8000 --to bore.pub')
print('  -> bore.pub:<PORT>')
